# 01 — Synthetic cluster light curves from stellar populations

Thin driver around `sythn_real_variability.run_generate`. See `README.md` for the construction,
the fixes relative to the Dropbox code, and the caveats.

**Two environments.** Stage 1 (populations) needs `isochrones` + the MIST grids → kernel `synth_pop`.
Stage 2 (variability, TESS sampling, truth, checks, figures) runs in the pipeline kernel `tess_gp`.
Without `isochrones`, stage 1 falls back to the Dropbox `isos/` tables (bootstrap-resampled, re-scaled to the LMC).

Equivalent shell commands (run from `GenerateTable_CGW/sytheticdata_generate_compare`):
```
python -m sythn_real_variability.run_generate populations     # env synth_pop, ~30 min for 45 x 15
python -m sythn_real_variability.run_generate lcs             # env tess_gp,   ~2 h for 675 clusters
python -m sythn_real_variability.checks
python -m sythn_real_variability.make_figs
```

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
MODULE_DIR = os.path.abspath('..')          # sytheticdata_generate_compare
for p in (MODULE_DIR, os.path.abspath(os.path.join(MODULE_DIR, '..'))):
    if p not in sys.path:
        sys.path.insert(0, p)

from sythn_real_variability.config import RunConfig
from sythn_real_variability import run_generate as R

cfg = RunConfig()                 # every construction parameter lives here
cfg.population.n_systems = 10_000
cfg.population.n_instances = 15
print('populations ->', cfg.paths.populations_dir)
print('light curves ->', cfg.paths.out_lc_dir)
print('truth table  ->', cfg.paths.truth_file)

## Stage 1 — stellar populations (kernel `synth_pop`)

MIST isochrones through `isochrones`: Salpeter IMF 0.1–300 M⊙, binary fraction 0.4, mass-ratio exponent 0.3,
delta-function SFH, Gaussian [Fe/H] (σ = 0.01), at the LMC distance (49.59 kpc). A fresh IMF draw per instance.
TESS magnitudes come from the MIST bolometric-correction grid, never from the (half-NaN) `TESS_mag` columns.

In [ ]:
# metas = R.run_populations(cfg)                       # full grid
metas = R.run_populations(cfg, ages=[8.0], fehs=[0.0], instances=[0])   # quick test

## Stage 2 — variability, TESS sampling, noise, truth (kernel `tess_gp`)

Each stellar component is binned on the empirical Teff × log L grid, is variable with the bin's variable fraction,
and copies the (A, P) of a real TF2023/TF2026 star of its bin. The integrated signal is exposure-averaged onto a
real LMC cluster's timestamps (brightness-matched template), given that template's `flux_err` noise and calibrated
red noise, and written as a FITS file the pipeline reads unchanged.

In [ ]:
# rows = R.run_lcs(cfg)                                # full grid
rows = R.run_lcs(cfg, ages=[8.0], fehs=[0.0], instances=[0])

## Checks and figures

In [ ]:
from sythn_real_variability.checks import run_checks
from sythn_real_variability.make_figs import make_all
run_checks(cfg)
make_all(cfg, ages=(7.0, 8.0, 9.5), feh=0.0, inst=0)

## Running the real pipeline on this set

In `generate_table.ipynb` replace the Wainer base table and the LC directory:
```python
import pandas as pd
base_table = pd.read_csv('sytheticdata_generate_compare/sythn_real_variability/data/base_table_synthetic_populations.csv')
LC_DIR = '../light_curves_synthetic_populations'
```
Names, `origin` (`LMC`) and file names round-trip through `lc_lsp.get_lc_path` (asserted at generation time).